In [ ]:

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


In [ ]:


data = []
# urls = []
# ids = []

TOP_GENRES = {'Drama', 'Action', 'Crime', 'Adventure', 'Sci-Fi', 'Horror'}

#Put the imdb genre page (Like this one >> 'https://www.imdb.com/search/title/?title_type=feature,tv_movie,tv_special,video&interests=in0000076' ).

url = ''
base_link = 'https://www.imdb.com'

#Put your target(how many movies) to fetch.
TARGET = ...

driver = webdriver.Chrome()
driver.get(url)
time.sleep(3)

all_hrefs = []

while len(all_hrefs) < TARGET:
    html = driver.page_source
    soup_main = BeautifulSoup(html, 'lxml')
    movies = soup_main.find_all('li', class_="ipc-metadata-list-summary-item")

    for movie in movies:
        link_tag = movie.find('a', class_='ipc-lockup-overlay')
        if link_tag and link_tag['href'] not in all_hrefs:
            all_hrefs.append(link_tag['href'])

    if len(all_hrefs) >= TARGET:
        break

    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        see_more_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button.ipc-see-more__button'))
        )
        driver.execute_script("arguments[0].click();", see_more_btn)
        time.sleep(3)
    except Exception as e:
        print(f"No more pages: {e}")
        break

all_hrefs = all_hrefs[:TARGET]

for i, href in enumerate(all_hrefs):
    movie_page = base_link + href
    # movie_id = href.split("title/")[-1].split("/")[0]

    driver.get(movie_page)
    time.sleep(2)
    soup_sub = BeautifulSoup(driver.page_source, 'lxml')

    # Title
    title_tag = soup_sub.find('span', {'data-testid': 'hero__primary-text'})
    if title_tag: title = title_tag.text.strip()  
    else: None

    # Description
    desc_tag = soup_sub.find('span', {'data-testid': 'plot-xs_to_m'})
    if desc_tag:
        description = desc_tag.text.strip()  
    else :None

    # Genre
    genre_section = soup_sub.find('div', {'data-testid': 'interests'})
    genres = [a.text.strip() for a in genre_section.find_all('a')] if genre_section else []
    filtered_genres = [g for g in genres if g in TOP_GENRES]

    # Stars
    sections = soup_sub.find_all('li', {'data-testid': 'title-pc-principal-credit'})
    stars = []
    if sections:
        stars = [a.text.strip() for a in sections[-1].find_all('a')
                 if a.text.strip() and a.text.strip() != "Stars"]

    poster_url = None
    poster_link = soup_sub.select_one('a[data-testid="hero-media__poster"]')
    
    if not poster_link:
       
        poster_link = soup_sub.select_one('div[data-testid="hero-media__poster"] a')

    if poster_link:
        full_poster_url = base_link + poster_link['href']
        driver.get(full_poster_url)
        time.sleep(2)
        soup_poster = BeautifulSoup(driver.page_source, 'lxml')
        img = soup_poster.select_one('div[data-testid="media-viewer"] img')
        if img:
            srcset = img.get('srcset', '')
            if srcset:
                # Take the highest resolution (last entry in srcset)
                poster_url = srcset.strip().split(',')[-1].strip().split(' ')[0]
            else:
                poster_url = img.get('src')


    # ids.append({'ID': movie_id})

    data.append({
        
        'Title': title,
        'Description': description,
        'Actors': stars,
        'Genre': filtered_genres,
        'poster_url' : poster_url
        
    })

    # urls.append({'poster_url': poster_url})
    
    print(f"{i + 1} > '{title}' is added")
    time.sleep(1)

driver.quit()
print(f"\ncollected: {len(data)}")

> Make sure to add the name of the csv for both movies and its posters' links

In [ ]:
data_ = pd.DataFrame(data)
# data_.to_csv(".csv", index=False)

# links_ = pd.DataFrame(urls)
# ds.to_csv(".csv", index=False)

data_.info()

In [ ]:
data_['Genre'].explode().value_counts()

In [ ]:
data_['Title'].str.strip().str.lower().nunique()
duplicates = []
duplicates.append(data_[data_['Title'].duplicated()]['Title'].values)
print(duplicates[0])
# duplicates[0].size

# data_[data_['Title'] == "Beast"]